# Employee Attrition — EDA Course Project (Phase 1)

**Dataset:** [Employee Attrition.csv](https://raw.githubusercontent.com/salemprakash/EDA/main/Data/Employee%20Attrition.csv)

**Phase 1 tasks covered in this notebook:**
1. Loading the dataset
2. Basic statistical analysis
3. Handling missing data
4. Data cleaning
5. Data transformation
6. Univariate analysis (≥ 3 visualizations)
7. Bivariate analysis (≥ 3 visualizations)
8. Multivariate analysis (≥ 3 visualizations)

> Run all cells top to bottom (Runtime → Run all). This notebook is self-contained and only needs an internet connection to fetch the dataset directly from GitHub.


## 0. Import Libraries

In [ ]:
# Core libraries for data handling and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

pd.set_option("display.max_columns", None)


## 1. Loading the Dataset

The dataset is read directly from the raw GitHub URL, so no manual upload is required.


In [ ]:
url = "https://raw.githubusercontent.com/salemprakash/EDA/main/Data/Employee%20Attrition.csv"
df = pd.read_csv(url)

print("Shape of dataset (rows, columns):", df.shape)
df.head()


In [ ]:
# Quick structural overview: column names, non-null counts, and data types
df.info()


## 2. Basic Statistical Analysis

Summary statistics for numeric and categorical columns to understand the central
tendency, spread, and distribution of the data.


In [ ]:
# Descriptive statistics for all NUMERIC columns
df.describe().T


In [ ]:
# Descriptive statistics for all CATEGORICAL (object) columns
df.describe(include="object").T


In [ ]:
# Target variable distribution (class balance check)
print(df["Attrition"].value_counts())
print()
print(df["Attrition"].value_counts(normalize=True).round(3) * 100, "%")


## 3. Handling Missing Data

Check for missing/null values across all columns and decide how to treat them.


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_summary = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct.round(2)})
missing_summary = missing_summary[missing_summary["Missing Count"] > 0].sort_values("Missing Count", ascending=False)

if missing_summary.empty:
    print("No missing values found in the dataset.")
else:
    print(missing_summary)


In [ ]:
# Defensive handling in case missing values ARE found when re-run on an updated dataset:
# - Numeric columns  -> fill with median (robust to outliers)
# - Categorical columns -> fill with mode (most frequent category)

num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include="object").columns

for c in num_cols:
    if df[c].isnull().sum() > 0:
        df[c] = df[c].fillna(df[c].median())

for c in cat_cols:
    if df[c].isnull().sum() > 0:
        df[c] = df[c].fillna(df[c].mode()[0])

print("Missing values remaining after handling:", df.isnull().sum().sum())


## 4. Data Cleaning

- Remove duplicate rows (if any)
- Drop columns that carry no analytical value (constant across all rows)
- Fix data types where appropriate


In [ ]:
# Check and remove duplicate rows
dup_count = df.duplicated().sum()
print("Number of duplicate rows:", dup_count)
df = df.drop_duplicates()


In [ ]:
# Identify columns with only ONE unique value — they add no information for EDA/modeling
constant_cols = [c for c in df.columns if df[c].nunique() == 1]
print("Constant (non-informative) columns:", constant_cols)

df_clean = df.drop(columns=constant_cols)
print("\nShape after dropping constant columns:", df_clean.shape)


In [ ]:
# Sanity check: unique values for a few categorical columns (helps spot inconsistent labels)
for c in ["Attrition", "BusinessTravel", "Department", "Gender", "OverTime"]:
    print(f"{c}: {df_clean[c].unique()}")


## 5. Data Transformation

- Encode the binary target (`Attrition`) as 0/1 for easier plotting and future modeling
- Create a few readable derived features (age groups, income in K) used later in the analysis
- Encode other Yes/No columns similarly


In [ ]:
# Binary-encode Yes/No columns
binary_map = {"Yes": 1, "No": 0}
df_clean["Attrition_Flag"] = df_clean["Attrition"].map(binary_map)
df_clean["OverTime_Flag"] = df_clean["OverTime"].map(binary_map)

# Feature engineering: bucket Age into readable groups
bins = [17, 25, 35, 45, 55, 65]
labels = ["18-25", "26-35", "36-45", "46-55", "56-65"]
df_clean["AgeGroup"] = pd.cut(df_clean["Age"], bins=bins, labels=labels)

# Feature engineering: Monthly income expressed in thousands (easier to read on plots)
df_clean["MonthlyIncome_K"] = (df_clean["MonthlyIncome"] / 1000).round(1)

df_clean[["Age", "AgeGroup", "MonthlyIncome", "MonthlyIncome_K", "Attrition", "Attrition_Flag", "OverTime", "OverTime_Flag"]].head()


In [ ]:
# Final cleaned & transformed dataset used for the rest of the analysis
df_final = df_clean.copy()
df_final.shape


## 6. Univariate Analysis
Examining the distribution of individual variables (minimum 3 visualizations).


### 6.1 Distribution of Employee Age

In [ ]:
plt.figure()
sns.histplot(df_final["Age"], bins=20, kde=True, color="steelblue")
plt.title("Distribution of Employee Age")
plt.xlabel("Age")
plt.ylabel("Number of Employees")
plt.show()


### 6.2 Attrition Count (Target Variable)

In [ ]:
plt.figure()
sns.countplot(data=df_final, x="Attrition", palette="Set2")
plt.title("Employee Attrition Count")
plt.xlabel("Attrition")
plt.ylabel("Number of Employees")
plt.show()


### 6.3 Spread of Monthly Income

In [ ]:
plt.figure()
sns.boxplot(data=df_final, x="MonthlyIncome", color="orange")
plt.title("Boxplot of Monthly Income")
plt.xlabel("Monthly Income")
plt.show()


### 6.4 Job Role Frequency (bonus)

In [ ]:
plt.figure(figsize=(10, 5))
order = df_final["JobRole"].value_counts().index
sns.countplot(data=df_final, y="JobRole", order=order, palette="viridis")
plt.title("Number of Employees by Job Role")
plt.xlabel("Number of Employees")
plt.ylabel("Job Role")
plt.show()


## 7. Bivariate Analysis
Examining relationships between pairs of variables (minimum 3 visualizations).


### 7.1 Monthly Income vs Attrition

In [ ]:
plt.figure()
sns.boxplot(data=df_final, x="Attrition", y="MonthlyIncome", palette="Set3")
plt.title("Monthly Income by Attrition Status")
plt.xlabel("Attrition")
plt.ylabel("Monthly Income")
plt.show()


### 7.2 OverTime vs Attrition

In [ ]:
plt.figure()
sns.countplot(data=df_final, x="OverTime", hue="Attrition", palette="Set1")
plt.title("Attrition Count by OverTime Status")
plt.xlabel("OverTime")
plt.ylabel("Number of Employees")
plt.show()


### 7.3 Age vs Monthly Income

In [ ]:
plt.figure()
sns.scatterplot(data=df_final, x="Age", y="MonthlyIncome", hue="Attrition", alpha=0.6)
plt.title("Age vs Monthly Income (colored by Attrition)")
plt.xlabel("Age")
plt.ylabel("Monthly Income")
plt.show()


### 7.4 Job Satisfaction vs Attrition (bonus)

In [ ]:
plt.figure()
sns.countplot(data=df_final, x="JobSatisfaction", hue="Attrition", palette="coolwarm")
plt.title("Job Satisfaction Level by Attrition")
plt.xlabel("Job Satisfaction (1=Low, 4=Very High)")
plt.ylabel("Number of Employees")
plt.show()


## 8. Multivariate Analysis
Examining relationships among three or more variables at once (minimum 3 visualizations).


### 8.1 Correlation Heatmap of Numeric Features

In [ ]:
plt.figure(figsize=(14, 10))
numeric_df = df_final.select_dtypes(include=np.number)
corr = numeric_df.corr()
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.3)
plt.title("Correlation Heatmap of Numeric Features")
plt.show()


### 8.2 Monthly Income by Department and Attrition

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df_final, x="Department", y="MonthlyIncome", hue="Attrition", palette="Set2")
plt.title("Monthly Income by Department and Attrition Status")
plt.xlabel("Department")
plt.ylabel("Monthly Income")
plt.xticks(rotation=15)
plt.show()


### 8.3 Age, Monthly Income and Attrition (pairwise relationships)

In [ ]:
subset_cols = ["Age", "MonthlyIncome", "TotalWorkingYears", "JobSatisfaction", "Attrition"]
sns.pairplot(df_final[subset_cols], hue="Attrition", diag_kind="kde", corner=True)
plt.suptitle("Pairwise Relationships Colored by Attrition", y=1.02)
plt.show()


### 8.4 Attrition Rate by Department and Gender (bonus)

In [ ]:
grouped = (
    df_final.groupby(["Department", "Gender"])["Attrition_Flag"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index(name="AttritionRate(%)")
)

plt.figure(figsize=(9, 5))
sns.barplot(data=grouped, x="Department", y="AttritionRate(%)", hue="Gender", palette="pastel")
plt.title("Attrition Rate (%) by Department and Gender")
plt.ylabel("Attrition Rate (%)")
plt.xticks(rotation=15)
plt.show()


## 9. Summary of Findings (Phase 1)

- The dataset contains **1470 employee records** with **35 original columns**; no missing values were found.
- Three constant columns (`EmployeeCount`, `StandardHours`, `Over18`) carried no analytical value and were dropped.
- Overall attrition rate is roughly **16%**, indicating class imbalance in the target variable.
- Employees who work **OverTime** show a noticeably higher attrition rate than those who don't.
- **Lower monthly income** and **lower job satisfaction** are visibly associated with higher attrition.
- The correlation heatmap highlights strong relationships between tenure-related features
  (e.g., `YearsAtCompany`, `YearsInCurrentRole`, `YearsWithCurrManager`).

*These observations will be explored further with statistical testing and modeling in later phases of the project.*
